In [1]:
import torch
import torch.nn as nn
import torch.functional as F
import numpy as np
import math

In [2]:
class SingleHeadAttentionBlock(nn.Module):

    def __init__(self, 
                 d_emb: int,
                 d_model: int,
                 causal: bool,
                 dropout_prob: float,
                 **kwargs):

        super().__init__()

        self.d_model = d_model
        self.d_emb = d_emb
        self.causal = causal,
        
        self.w_q = nn.Linear(d_emb, d_model, bias=False)
        self.w_k = nn.Linear(d_emb, d_model, bias=False)
        self.w_v = nn.Linear(d_emb, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_emb, bias=True)
        self.dropout = nn.Dropout(dropout_prob)

        self.softmax = nn.Softmax(dim=-1)
        self.norm = nn.LayerNorm(d_emb)

    def forward(self, x):

        Q = self.w_q(x) # [b, s, d_model]
        K = self.w_k(x) # [b, s, d_model]
        V = self.w_v(x) # [b, s, d_model]

        A = Q @ K.transpose(-1, -2) / math.sqrt(self.d_model) # [b, s, s]
        if self.causal:
            #TODO rethink about it
            s = Q.shape[1]
            causal_mask = torch.ones((s, s), device=x.device, dtype=bool).triu(1)
            A = A.masked_fill(causal_mask, float("-inf"))
        A = self.softmax(A) 
        A = self.dropout(A)

        S = A @ V # [b, s, d_model]
        O = self.w_o(S)
        O = self.norm(O)

        return x + O, A


In [3]:
class MultiheadAttentionBlock(nn.Module):

    def __init__(self,
                 d_model: int,
                 d_emb: int,
                 num_head: int,
                 causal: bool,
                 dropout: float,
                 **kwargs):

        super().__init__()

        self.d_model = d_model
        self.d_emb = d_emb
        self.num_head = num_head
        self.causal = causal

        assert d_model % num_head == 0
        self.d_head = d_model // num_head

        #TODO the qkv can be constructed in one
        self.w_q = nn.Linear(d_emb, d_model, bias=False)
        self.w_k = nn.Linear(d_emb, d_model, bias=False)
        self.w_v = nn.Linear(d_emb, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_emb, bias=False)

        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_emb)

    def forward(self, x):

        s = x.size(1)
        Q = self.w_q(x) # [b, s, d_model]
        Q = Q.view((-1, s, self.d_head, self.num_head)).permute(0, 3, 1, 2) #[b, s, d_head, num_head] -> [b, num_head, s, d_head]
        K = self.w_k(x) # [b, s, d_model]
        K = K.view((-1, s,self.d_head, self.num_head)).permute(0, 3, 1, 2) #[b, s, d_head, num_head] -> [b, num_head, s, d_head]

        A = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head) # [b, num_head, s, s]
        if self.causal:
            mask = torch.ones((s, s), dtype=bool, device=x.device).triu(1)
            A = A.masked_fill(mask, float("inf"))
        A = self.softmax(A)
        V = self.w_v(x) # [b, s, d_model]
        V = V.view((-1, s, self.d_head, self.num_head)).permute(0, 3, 1, 2) #[b, s, d_head, num_head] -> [b, num_head, s, d_head]

        S = A @ V # [b, num_head, s, d_head]
        S = S.permute(0, 2, 1, 3).view((-1, s, self.d_model))  #[b, s, d_head, num_head] -> [b, s, d_model]

        O = self.w_o(S)
        O = self.norm(O)

        return x + O
